# 终极 AI Agent 记忆实验室：端到端实现

### 一个全面的、实践性的工作坊，用于构建更智能的 Agent

欢迎来到对 AI Agent 记忆的详细、实用探索。 本笔记本旨在成为一份确定的、端到端的指南，超越理论，进入可工作的代码。 我们将使用真实的大语言模型进行生成、摘要和嵌入，实现**九种不同的记忆策略**，从最简单到最具概念性的高级方法。

**目标：** 在本实验结束时，你将拥有深入、实用的理解：
- 每种记忆策略如何运作。
- 每种方法的具体优势、劣势和权衡。
- 如何使用 `openai`、`faiss-cpu` 和 `networkx` 等现代工具在 Python 中实现这些策略。
- 记忆架构的选择如何从根本上改变 Agent 的对话能力、成本和复杂性。

**实验室结构：**
1.  **第一部分：核心框架。** 我们将设置环境、配置 LLM 客户端，并构建基础的 `AIAgent` 和 `BaseMemoryStrategy` 类。
2.  **第二部分：记忆实现。** 我们将系统地实现并演示所有九种记忆策略。 每种策略都有专门的部分，包含：
    *   **详细理论：** 解释*什么*、*为什么*和*如何*。
    *   **代码实现：** 一个完整的、有中文注释的 Python 类。
    *   **实际演示：** 一个实际聊天会话，旨在展示策略的独特行为。

本笔记本故意冗长而详细，以作为全面的参考。让我们开始设置核心组件。

## 第一部分：核心框架和设置

在我们构建记忆之前，我们需要一个大脑（LLM）和一个身体（Agent 框架）。本节处理所有初步设置。

### 步骤 1.0：创建虚拟环境

使用 conda 创建一个虚拟环境。然后在 Notebook 切换 Kernel：

```text
Kernel -> Change Kernel -> test_env
```

In [ ]:
%conda create -n env_test python=3.12 -y
%conda run -n env_test pip install ipykernel
%conda run -n env_test python -m ipykernel install --user --name=env_test

### 步骤 1.1：安装依赖

首先，我们需要安装必要的 Python 库。我们需要：
- `openai`：用于与 LLM API 交互的客户端库。
- `numpy`：用于数值操作，特别是嵌入计算。
- `faiss-cpu`：来自 Facebook AI 的高效相似度搜索库，它为我们的检索记忆提供动力。它是一个完美的内存向量数据库。
- `networkx`：用于在我们的基于图的记忆策略中创建和管理知识图谱。
- `tiktoken`：用于精确计算 token 和管理上下文窗口限制。

In [ ]:
%pip install openai numpy faiss-cpu networkx tiktoken

### 步骤 1.2：配置 LLM 和嵌入客户端

在这里，我们将使用你提供的自定义 `base_url` 和 `api_key` 设置 `OpenAI` 客户端。这个单一客户端将用于文本生成和创建嵌入。

In [ ]:
# 导入必须的库
import os
from openai import OpenAI


# --- 重要：API 密钥配置 ---
# 为简单起见，API 密钥直接在这里提供。
# 在生产环境中，切勿硬编码密钥。使用环境变量
# 或安全的密钥管理服务。

# 定义用于认证的 API 密钥。
API_KEY = os.getenv("OPENAI_API_KEY")
# 定义 API 端点的 base URL。
BASE_URL = os.getenv("OPENAI_BASE_URL")

# 使用指定的 base URL 和 API 密钥初始化 OpenAI 客户端。
client = OpenAI(
    base_url=BASE_URL,
    api_key=API_KEY
)

try:
    models = client.models.list()
    print("连接成功！可用模型：")
    for m in models.data[:5]:
        print(m.id)
except Exception as e:
    print("连接失败：", e)
    
# 打印确认消息以指示客户端设置成功。
print("OpenAI 客户端配置成功。")

### 步骤 1.3：LLM 交互和 Token 计数的辅助函数

为保持主 Agent 逻辑清晰，我们将为 API 调用创建包装函数。我们还将设置一个 token 计数器，这对于理解记忆策略的成本和限制至关重要。

In [ ]:
# 导入额外的功能性库
import tiktoken
import time

# --- 模型配置 ---
# 定义用于生成任务和 embedding 任务的具体模型
GENERATION_MODEL = os.getenv("GENERATION_MODEL")
EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL")

def generate_text(system_prompt: str, user_prompt: str) -> str:
    """
    调用 LLM API 生成文本回复
    
    参数:
        system_prompt: 用于定义 AI 角色和行为的系统指令
        user_prompt: 用户输入内容
        
    返回:
        AI 生成的文本内容，如果出错则返回错误信息
    """
    try:
        # 调用聊天模型接口生成回复
        response = client.chat.completions.create(
            model=GENERATION_MODEL,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ]
        )
        # 提取并返回模型生成的文本内容
        return response.choices[0].message.content
    except Exception as e:
        # 捕获并处理 API 调用错误
        print(f"文本生成过程中发生错误: {e}")
        return "抱歉，我在处理您的请求时遇到了错误。"

def generate_embedding(text: str) -> list[float]:
    """
    使用 embedding 模型将文本转换为向量表示
    
    参数:
        text: 需要转换为向量的文本
        
    返回:
        向量列表（浮点数），如果失败则返回空列表
    """
    try:
        # 调用 embedding 接口
        response = client.embeddings.create(
            model=EMBEDDING_MODEL,
            input=text
        )
        # 提取 embedding 向量
        return response.data[0].embedding
    except Exception as e:
        # 捕获并处理 API 错误
        print(f"生成 embedding 时发生错误: {e}")
        return []

# --- token 计数设置 ---
# 初始化 tokenizer，使用 tiktoken
# 'cl100k_base' 是一种常见编码方式，广泛用于 OpenAI 和 LLaMA 等模型
# 用于在发送 prompt 前精确估算 token 数量
tokenizer = tiktoken.get_encoding("cl100k_base")

def count_tokens(text: str) -> int:
    """
    统计输入文本的 token 数量
    
    参数:
        text: 需要分词的字符串
        
    返回:
        token 数量（整数）
    """
    # encode 方法将文本转换为 token ID 列表
    # 列表长度即为 token 数
    return len(tokenizer.encode(text))

# 输出提示信息，表示核心函数已准备就绪
print("辅助函数和 token 计数器已准备就绪。")

### 步骤 1.4：基础 Agent 和记忆类

现在我们使用策略设计模式定义系统的核心结构。

- **`BaseMemoryStrategy`**：一个定义任何记忆类型通用接口的抽象基类。 每个策略都继承自此，确保它可以无缝插入我们的 Agent。
- **`AIAgent`**：Agent 类本身。它使用记忆策略初始化。它的 `chat` 方法协调从记忆获取上下文、查询 LLM 和更新记忆的过程。

In [ ]:
import abc

# --- 抽象基类：记忆策略 ---
# 这个类定义了所有记忆策略必须遵循的“接口/契约”
# 通过抽象基类（ABC），我们确保所有记忆实现都必须包含相同的核心方法
# （add_message、get_context、clear），从而可以在 AIAgent 中自由替换
class BaseMemoryStrategy(abc.ABC):
    """所有记忆策略的抽象基类"""
    
    @abc.abstractmethod
    def add_message(self, user_input: str, ai_response: str):
        """
        抽象方法，子类必须实现
        用于将一次用户-AI对话写入记忆存储
        """
        pass

    @abc.abstractmethod
    def get_context(self, query: str) -> str:
        """
        抽象方法，子类必须实现
        从记忆中检索并格式化相关上下文，用于发送给大模型
        query 参数允许某些策略（如检索式记忆）根据用户输入获取相关内容
        """
        pass

    @abc.abstractmethod
    def clear(self):
        """
        抽象方法，子类必须实现
        用于清空记忆，常用于开启新对话
        """
        pass


# --- 核心 AI Agent ---
# 该类负责整体对话流程的编排
# 初始化时注入一个具体的记忆策略，用于管理上下文
class AIAgent:
    """核心 AI Agent，实现与任意记忆策略解耦"""
    
    def __init__(self, memory_strategy: BaseMemoryStrategy, system_prompt: str = "你是一个有帮助的AI助手。"):
        """
        初始化 Agent
        
        参数:
            memory_strategy: BaseMemoryStrategy 的子类实例
                             决定 Agent 使用哪种记忆机制
            system_prompt: 系统提示词，用于定义模型角色和行为
        """
        self.memory = memory_strategy
        self.system_prompt = system_prompt
        print(f"Agent 初始化完成，使用记忆策略: {type(memory_strategy).__name__}")

    def chat(self, user_input: str):
        """
        处理一次完整对话流程
        
        参数:
            user_input: 用户输入
        """
        print(f"\n{'='*25} 新对话 {'='*25}")
        print(f"User > {user_input}")
        
        # Step 1: 从记忆策略中获取上下文
        # 这里执行具体的记忆检索逻辑（如顺序记忆 / 向量检索）
        start_time = time.time()
        context = self.memory.get_context(query=user_input)
        retrieval_time = time.time() - start_time
        
        # Step 2: 构造 LLM 输入 prompt
        # 将历史记忆 + 当前问题拼接成完整输入
        full_user_prompt = f"### 记忆上下文\n{context}\n\n### 当前请求\n{user_input}"
        
        # Step 3: 调试信息输出
        # 用于分析 prompt token、成本、延迟等
        prompt_tokens = count_tokens(self.system_prompt + full_user_prompt)
        print("\n--- 调试信息 ---")
        print(f"记忆检索耗时: {retrieval_time:.4f} 秒")
        print(f"估算 Prompt Token 数: {prompt_tokens}")
        print(f"\n[发送给 LLM 的完整 Prompt]:\n---\nSYSTEM: {self.system_prompt}\nUSER: {full_user_prompt}\n---")
        
        # Step 4: 调用 LLM 生成回复
        start_time = time.time()
        ai_response = generate_text(self.system_prompt, full_user_prompt)
        generation_time = time.time() - start_time
        
        # Step 5: 更新记忆
        # 将本轮对话写入 memory
        self.memory.add_message(user_input, ai_response)
        
        # Step 6: 输出结果
        print(f"\nAgent > {ai_response}")
        print(f"(LLM 生成耗时: {generation_time:.4f} 秒)")
        print(f"{'='*70}")

## 第二部分：记忆策略的实现与演示

这是我们实验室的核心内容。现在我们将逐一实现所有九种记忆策略，并立即进行实际演示以观察它们在实践中的表现。

### 策略 1：顺序（保留全部）记忆

| **最佳用于**              | **权衡**                                              |
| ----------------------------- | ------------------------------------------------------ |
| 短期交互、完整保真度              | 快速达到 token 限制，成本高昂                           |

**理论：** 这是最直接的记忆类型。 它简单地将每个用户-AI 交互追加到不断增长的列表中。 生成新响应时，整个对话历史被格式化并作为上下文发送给 LLM。 这保证了在单个对话会话内的完美、无损回忆。

然而，其简单性在长对话中成为它的致命弱点。 上下文随着每轮线性增长，导致 API 成本快速增加，最终超过 LLM 的最大上下文窗口，这将导致错误或截断的提示。

In [ ]:
# --- 策略 1: 顺序（保留全部）记忆 ---
# 这是最基础的记忆策略。它将整个对话历史存储在一个简单的列表中。
# 虽然提供了完美的回忆能力，但随着每次交互上下文的增长，
# 成本会快速增加，很快就会达到 token 限制，无法规模化。
class SequentialMemory(BaseMemoryStrategy):
    def __init__(self):
        """使用空列表初始化记忆，用于存储对话历史。"""
        self.history = []

    def add_message(self, user_input: str, ai_response: str):
        """
        将新的用户-AI 交互添加到历史记录中。
        每次交互作为两个字典条目存储在列表中。
        """
        self.history.append({"role": "user", "content": user_input})
        self.history.append({"role": "assistant", "content": ai_response})

    def get_context(self, query: str) -> str:
        """
        检索整个对话历史并将其格式化为单个字符串，
        用作 LLM 的上下文。'query' 参数在此策略中被忽略，
        因为它总是返回完整的对话历史。
        """
        # 将所有消息连接成一个用换行符分隔的字符串。
        return "\n".join([f"{turn['role'].capitalize()}: {turn['content']}" for turn in self.history])

    def clear(self):
        """通过清空列表来重置对话历史。"""
        self.history = []
        print("顺序记忆已清空。")

#### 顺序记忆演示

**观察重点：** 密切关注调试输出中的 `估计的提示 Token 数`。 你会看到它随着每轮增加，因为整个历史被添加到提示中。

In [ ]:
# 初始化并运行 Agent
# 创建 SequentialMemory 策略实例（顺序记忆）
sequential_memory = SequentialMemory()

# 创建 AIAgent，并注入顺序记忆策略
agent = AIAgent(memory_strategy=sequential_memory)

# --- 开始对话 ---

# 第一轮：用户介绍自己
agent.chat("你好！我叫 Sam。")

# 第二轮：用户说明兴趣
agent.chat("我对学习太空探索很感兴趣。")

# 第三轮：测试 Agent 的记忆能力
agent.chat("我最开始告诉你的是什么？")

# 由于顺序记忆会保留完整历史上下文
# Agent 具备“完全回忆能力”

# 清理记忆，用于下一次演示
sequential_memory.clear()

# -------------------------（重复了一遍同样流程）-------------------------

# 初始化并运行 Agent
# 创建 SequentialMemory 策略实例（顺序记忆）
sequential_memory = SequentialMemory()

# 创建 AIAgent，并注入顺序记忆策略
agent = AIAgent(memory_strategy=sequential_memory)

# --- 开始对话 ---

# 第一轮：用户介绍自己
agent.chat("你好！我叫 Sam。")

# 第二轮：用户说明兴趣
agent.chat("我对学习太空探索很感兴趣。")

# 第三轮：测试 Agent 的记忆能力
agent.chat("我最开始告诉你的是什么？")

# 由于顺序记忆会保留完整历史上下文
# Agent 具备“完全回忆能力”

# 清理记忆，用于下一次演示
sequential_memory.clear()

### 策略 2：滑动窗口记忆

| **最佳用于**              | **权衡**                                              |
| ----------------------------- | ------------------------------------------------------ |
| 中等长度聊天、最近相关性重要            | 会遗忘早期上下文                           |

**理论：** 此策略通过只保留最近的 `N` 个对话轮次来解决顺序记忆的主要问题。 它使用具有固定最大长度的 `deque`（双端队列）。 当添加新交互且 deque 已满时，最旧的交互会自动被丢弃。

这使上下文大小保持恒定，使成本可预测并防止上下文窗口溢出。 主要权衡是健忘症：在窗口截止点之前提到的任何信息都会被永久遗忘。

In [ ]:
# 从 collections 模块导入 deque 类。deque 是双端队列，
# 在两端添加和删除元素时效率很高。
from collections import deque

# --- 策略 2: 滑动窗口记忆 ---
# 此策略只保留最近 N 次对话交互。
# 它防止上下文无限增长，使其具有可扩展性和成本效益，
# 但代价是遗忘旧信息。
class SlidingWindowMemory(BaseMemoryStrategy):
    def __init__(self, window_size: int = 4): # window_size 是交互次数（用户 + AI = 1 次交互）
        """
        使用固定大小的 deque 初始化记忆。

        Args:
            window_size: 记忆中保留的对话交互次数。
                        一次交互包含一条用户消息和一条 AI 响应。
        """
        # 带有 'maxlen' 的 deque 在添加新元素且已满时，会自动丢弃最旧的元素。
        # 这是滑动窗口的核心机制。我们按交互存储，所以 maxlen 是 window_size。
        self.history = deque(maxlen=window_size)

    def add_message(self, user_input: str, ai_response: str):
        """
        将新的对话交互添加到历史记录中。如果 deque 已满，
        最旧的交互会自动被移除。
        """
        # 每次交互（用户输入 + AI 响应）作为单个元素存储在 deque 中。
        # 这样可以通过交互次数轻松管理窗口大小。
        self.history.append([
            {"role": "user", "content": user_input},
            {"role": "assistant", "content": ai_response}
        ])

    def get_context(self, query: str) -> str:
        """
        检索窗口内的对话历史并将其格式化为单个字符串。'query' 参数被忽略。
        """
        # 创建一个临时列表来存放格式化的消息。
        context_list = []
        # 遍历 deque 中存储的每次交互。
        for turn in self.history:
            # 遍历该交互中的用户和助手消息。
            for message in turn:
                # 格式化消息并将其添加到列表中。
                context_list.append(f"{message['role'].capitalize()}: {message['content']}")
        # 将所有格式化的消息连接成单个字符串，用换行符分隔。
        return "\n".join(context_list)

    def clear(self):
        """通过清空 deque 来重置对话历史。"""
        self.history.clear()
        print("滑动窗口记忆已清空。")

#### 滑动窗口记忆演示

**观察重点：** 我们将设置窗口大小为 2 轮。 第三轮后，非常第一条信息（用户的名字）将被推出上下文窗口。 然后 Agent 将无法回忆起它。

In [ ]:
# 初始化滑动窗口记忆，窗口大小为2轮
# 这意味着 Agent 只会记住最近的两轮用户-AI交互
sliding_memory = SlidingWindowMemory(window_size=2)

# 创建 AIAgent，并注入滑动窗口记忆策略
agent = AIAgent(memory_strategy=sliding_memory)

# --- 开始对话 ---

# 第一轮：用户介绍自己（这是第1轮）
agent.chat("我叫 Priya，是一名软件开发工程师。")

# 第二轮：用户补充信息（此时记忆中保留第1轮和第2轮）
agent.chat("我主要使用 Python 和云计算技术。")

# 第三轮：用户提到爱好（加入后，第1轮会被挤出窗口）
# 因为窗口大小为2，所以现在只保留第2轮和第3轮
agent.chat("我最喜欢的爱好是徒步旅行。")

# 现在提问关于最开始的信息
# 此时发送给 LLM 的上下文只包含 Python / 云计算 和 徒步旅行
# 关于名字的信息（Priya）已经被遗忘

agent.chat("我叫什么名字？")

# Agent 很可能无法正确回答，因为第1轮已经被移出窗口

# 清理记忆，用于下一次演示
sliding_memory.clear()

### 策略 3：摘要记忆

| **最佳用于**              | **权衡**                                              |
| ----------------------------- | ------------------------------------------------------ |
| 长对话、需要一般上下文              | 可能会丢失细节                           |

**理论：** 此策略试图两全其美。 它不是简单丢弃旧消息，而是使用 LLM 本身定期创建对话的运行摘要。 它维护一个最近消息的临时缓冲区。 一旦缓冲区达到一定大小，其内容被摘要并与之前的摘要合并。

发送给 LLM 的上下文是 `running_summary` 和 `current_buffer` 的组合。 这使上下文大小保持可管理，同时保留整个对话的要点。 主要风险是信息丢失：如果 LLM 的摘要遗漏了关键但微妙的细节，该细节将永远丢失。

In [ ]:
# --- 策略 3: 摘要记忆 ---
# 此策略旨在通过定期摘要来管理长对话。
# 它保留一个近期消息的缓冲区。当缓冲区达到一定大小时，
# 它使用 LLM 调用将缓冲区内容与运行摘要合并。
# 这使上下文大小保持可管理，同时保留对话的要点。
# 主要风险是信息丢失——如果摘要不完美，则会造成信息丢失。
class SummarizationMemory(BaseMemoryStrategy):
    def __init__(self, summary_threshold: int = 4): # 默认值：4 条消息后进行摘要（2 次交互）
        """
        初始化摘要记忆。

        Args:
            summary_threshold: 在触发摘要操作之前累积的消息数
                             （用户 + AI 消息）。
        """
        # 存储到目前为止对话的持续更新的摘要。
        self.running_summary = ""
        # 一个临时列表，用于保存待摘要的近期消息。
        self.buffer = []
        # 触发摘要过程的阈值。
        self.summary_threshold = summary_threshold

    def add_message(self, user_input: str, ai_response: str):
        """
        将新的用户-AI 交互添加到缓冲区。如果缓冲区大小达到阈值，
        则触发记忆整合过程。
        """
        # 将最新的用户和 AI 消息追加到临时缓冲区。
        self.buffer.append({"role": "user", "content": user_input})
        self.buffer.append({"role": "assistant", "content": ai_response})

        # 检查缓冲区是否已满。
        if len(self.buffer) >= self.summary_threshold:
            # 如果已满，调用方法来摘要缓冲区内容。
            self._consolidate_memory()

    def _consolidate_memory(self):
        """
        使用 LLM 来摘要缓冲区内容并将其与现有的运行摘要合并。
        """
        print("\n--- [记忆整合已触发] ---")
        # 将缓冲消息列表转换为单个格式化的字符串。
        buffer_text = "\n".join([f"{msg['role'].capitalize()}: {msg['content']}" for msg in self.buffer])

        # 为 LLM 构建一个特定的提示词以执行摘要任务。
        # 它提供现有摘要和新的对话文本，要求生成一个单一的、更新后的摘要。
        summarization_prompt = (
            f"你是一名总结专家。你的任务是对一段对话进行简洁的总结。"
            f"将“之前的总结”和“新的对话内容”合并为一个更新后的总结。"
            f"请保留所有关键事实、姓名以及重要决策。\n\n"
            f"### 之前的总结：\n{self.running_summary}\n\n"
            f"### 新的对话内容：\n{buffer_text}\n\n"
            f"### 更新后的总结："
        )

        # 使用特定系统提示词调用 LLM 以获得新的摘要。
        new_summary = generate_text("你是一个专业的 summarization engine。", summarization_prompt)
        # 用新生成的整合后的摘要替换旧摘要。
        self.running_summary = new_summary
        # 清空缓冲区，因为其内容现在已被整合到摘要中。
        self.buffer = []
        print(f"--- [新摘要: '{self.running_summary}'] ---")

    def get_context(self, query: str) -> str:
        """
        构建要发送给 LLM 的上下文。它将长期运行摘要
        与短期缓冲区消息相结合。'query' 参数被忽略，
        因为此策略提供通用上下文。
        """
        # 格式化当前缓冲区中的消息。
        buffer_text = "\n".join([f"{msg['role'].capitalize()}: {msg['content']}" for msg in self.buffer])
        # 返回历史摘要和最新未摘要消息的组合上下文。
        return f"### 过去对话的摘要:\n{self.running_summary}\n\n### 近期消息:\n{buffer_text}"

    def clear(self):
        """通过清空摘要和缓冲区来重置记忆。"""
        self.running_summary = ""
        self.buffer = []
        print("摘要记忆已清空。")

#### 摘要记忆演示

**观察重点：** 在第二轮后（因为我们的阈值是 4 条消息）会出现 `[记忆整合已触发]` 消息。 后续轮的上下文将包含新的 AI 生成的摘要。 我们将看到 Agent 是否能回忆起第一轮中的细节，这些细节现在只存在于摘要中。

In [ ]:
# 初始化 SummarizationMemory（摘要记忆），阈值为4条消息（2轮对话）
# 这意味着在第2轮完整交互后会触发一次“记忆压缩/总结”
summarization_memory = SummarizationMemory(summary_threshold=4)

# 创建 AIAgent，并注入摘要记忆策略
agent = AIAgent(memory_strategy=summarization_memory)

# --- 开始对话 ---

# 第一轮：用户提供初始信息
agent.chat("我正在创办一家新公司，名字叫 Innovatech。我们的方向是可持续能源。")

# 第二轮：用户补充更具体的信息
# 在这一轮 AI 回复后，buffer 会累计到 4 条消息
# 触发记忆压缩（summary generation）
agent.chat("我们的第一个产品是智能太阳能板，代号 'Project Helios'。")

# 第三轮：继续补充信息
# 此时早期对话已经被压缩成 running summary
agent.chat("市场营销预算设定为 5 万美元。")

# 第四轮：测试 Agent 的记忆能力
# 此时上下文 = “摘要 + 最新对话”
agent.chat("我的公司叫什么？第一个产品是什么？")

# Agent 是否能正确回答，完全取决于摘要质量

# 清理记忆，用于下一次演示
summarization_memory.clear()

### 策略 4：基于检索的记忆

| **最佳用于**              | **权衡**                                              |
| ----------------------------- | ------------------------------------------------------ |
| 长期记忆、需要精确性              | 实现难度较大，需要向量数据库 + 排序         |

**理论：** 这是一个强大且广泛使用的策略，构成了检索增强生成（RAG）的基础。 不再线性保存对话历史，而是将每个信息单元（如每一轮对话）作为独立文档，存储在可检索的数据库中。 当用户提出新问题时，系统：
1. 将用户的查询转换为数值向量（嵌入）。
2. 搜索数据库找到 `k` 个最语义相似的文档嵌入。
3. 检索这些文档的原始文本。
4. 将检索到的文本注入 LLM 的上下文。

这允许 Agent 从过去的任何时间点提取相关信息，无论多久。 我们使用 `faiss` 创建高效的内存向量索引。

In [ ]:
# 导入数值操作和相似性搜索所需的库。
import numpy as np
import faiss

# --- 策略 4: 基于检索的记忆 ---
# 此策略将每次对话内容视为可搜索数据库中的文档。
# 它使用向量嵌入来查找和检索过去在语义上最相关的信息片段，
# 以回答新的查询。这是检索增强生成（RAG）的核心概念。
class RetrievalMemory(BaseMemoryStrategy):
    def __init__(self, k: int = 2, embedding_dim: int = 3584):
        """
        初始化检索记忆系统。

        Args:
            k: 对于给定查询要检索的 top 相关文档数量。
            embedding_dim: 嵌入模型生成的向量维度。
                          对于 BAAI/bge-multilingual-gemma2，是 3584。
        """
        # 要检索的最近邻数量。
        self.k = k
        # 嵌入向量的维度。必须与模型的输出匹配。
        self.embedding_dim = embedding_dim
        # 用于存储每个文档原始文本内容的列表。
        self.documents = []
        # 初始化 FAISS 索引。IndexFlatL2 使用 L2（欧几里得）距离进行穷举搜索，
        # 对于中等数量的向量很有效。
        self.index = faiss.IndexFlatL2(self.embedding_dim)

    def add_message(self, user_input: str, ai_response: str):
        """
        将新的对话交互添加到记忆中。每次交互的每个部分（用户输入和 AI 响应）
        被单独嵌入和索引，以实现精细化的检索。
        """
        # 我们将每次交互的每个部分存储为单独的文档，以允许更精确的匹配。
        # 例如，查询可能类似于过去的用户陈述，但不同于同一交互中 AI 的响应。
        docs_to_add = [
            f"User said: {user_input}",
            f"AI responded: {ai_response}"
        ]
        for doc in docs_to_add:
            # 生成文档的数值向量表示。
            embedding = generate_embedding(doc)
            # 仅在成功创建嵌入时继续。
            if embedding:
                # 存储原始文本。该文档的索引将对应于
                # 其向量在 FAISS 索引中的索引。
                self.documents.append(doc)
                # FAISS 要求输入向量是 float32 类型的 2D numpy 数组。
                vector = np.array([embedding], dtype='float32')
                # 将向量添加到 FAISS 索引中，使其可搜索。
                self.index.add(vector)

    def get_context(self, query: str) -> str:
        """
        根据与用户查询的语义相似性，从记忆中找到最相关的 k 个文档。
        """
        # 如果索引没有向量，则无法搜索。
        if self.index.ntotal == 0:
            return "记忆中还没有信息。"

        # 将用户查询转换为嵌入向量。
        query_embedding = generate_embedding(query)
        if not query_embedding:
            return "无法处理检索查询。"

        # 将查询嵌入转换为 FAISS 所需的格式。
        query_vector = np.array([query_embedding], dtype='float32')

        # 执行搜索。'search' 返回到查询向量的 k 个最近邻的距离和索引。
        distances, indices = self.index.search(query_vector, self.k)

        # 使用返回的索引来检索原始文本文档。
        # 我们检查 `i != -1`，因为 FAISS 可以为无效索引返回 -1。
        retrieved_docs = [self.documents[i] for i in indices[0] if i != -1]

        if not retrieved_docs:
            return "在记忆中找不到任何相关信息。"

        # 将检索到的文档格式化为字符串，用作上下文。
        return "### 从记忆中检索到的相关信息:\n" + "\n---\n".join(retrieved_docs)

    def clear(self):
        """通过清空文档和 FAISS 索引来完全重置记忆。"""
        self.documents = []
        self.index.reset()
        print("检索记忆（文档和 FAISS 索引）已清空。")

#### 基于检索的记忆演示

**观察重点：** 我们将进行一个关于两个完全不同话题的对话：度假计划和软件项目。 然后，我们将询问关于度假的问题。 调试输出将显示只有相关的度假文件被检索并注入提示，完全忽略无关的项目讨论。

In [ ]:
# 初始化 RetrievalMemory，k=2 表示每次检索最相关的前2条文档
retrieval_memory = RetrievalMemory(k=2)

# 创建 AIAgent，并注入检索式记忆策略
agent = AIAgent(memory_strategy=retrieval_memory)

# --- 开始对话（混合主题）---

# 第一轮：讨论旅行计划（作为一条文档存入记忆）
agent.chat("我计划明年春天去日本旅行。")

# 第二轮：讨论软件项目（作为另一条独立文档存入记忆）
agent.chat("我的软件项目前端使用 React 框架。")

# 第三轮：补充旅行信息
agent.chat("在旅行期间我想去东京和京都。")

# 第四轮：补充项目信息
agent.chat("我的项目后端将使用 Django 构建。")

# --- 测试检索机制 ---

# 现在询问关于旅行的问题
# Agent 会将问题转为 embedding，然后在记忆中做语义搜索
# 应该会检索到与“日本 / 东京 / 京都”更相关的内容
# 而忽略 React / Django 等软件项目内容
agent.chat("我计划在旅行中去哪些城市？")

# Agent 应该返回东京和京都相关信息

# 清理记忆，供下一次演示使用
retrieval_memory.clear()

### 策略 5：记忆增强的 Transformer（概念模拟）

| **最佳用于**              | **权衡**                                              |
| ----------------------------- | ------------------------------------------------------ |
| 随时间推移的丰富、不断发展的上下文            | 高级模型，成本更高                           |

**理论：** 这是对*模型架构本身*的修改，无法在 Agent 层面完全实现。 然而，我们可以*模拟其行为*。 核心思想是模型可以访问一个特殊的压缩记忆空间（像"便利贴"），除了其正常的上下文窗口。 它学习将关键信息写入这些记忆槽，并在需要时从中读取。

**我们的模拟：** 我们将创建一个 `MemoryAugmentedMemory` 类。 在每轮之后，它将使用 LLM 决定是否有任何信息重要到足以成为"关键记忆"。 如果是，它将创建该事实的简洁摘要并存储在特殊的 `memory_tokens` 列表中。 最终上下文将是最近聊天的滑动窗口和这些重要的 `memory_tokens` 的组合。

In [ ]:
# --- 策略 5: 记忆增强记忆（模拟）---
# 此策略模拟记忆增强 Transformer 模型的行为。
# 它维护一个短期滑动窗口来存储近期对话，以及一个单独的
# "memory tokens" 列表，用于存储从对话中提取的重要事实。
# 使用 LLM 调用来决定一段信息是否足够重要，
# 以至于需要被转换为一个持久的记忆 token。
class MemoryAugmentedMemory(BaseMemoryStrategy):
    def __init__(self, window_size: int = 2):
        """
        初始化记忆增强系统。

        参数:
            window_size: 短期记忆中保留的近期交互次数。
        """
        # 使用 SlidingWindowMemory 实例来管理近期对话历史。
        self.recent_memory = SlidingWindowMemory(window_size=window_size)
        # 一个列表，用于存储特殊的、持久的"便利贴"或关键事实。
        self.memory_tokens = []

    def add_message(self, user_input: str, ai_response: str):
        """
        将最新的交互添加到近期记忆，然后使用 LLM 调用来决定
        是否需要从这个交互中创建一个新的持久记忆 token。
        """
        # 首先，将新的交互添加到短期滑动窗口记忆中。
        self.recent_memory.add_message(user_input, ai_response)

        # 构建一个提示词，供 LLM 分析对话交互并
        # 决定它是否包含值得长期记忆的核心事实。
        fact_extraction_prompt = (
            f"分析以下对话内容。这一轮对话是否包含需要长期记忆的核心事实、偏好或决策？"
            f"例如：用户偏好（“我讨厌坐飞机”）、关键决策（“预算是1000美元”）、重要事实（“我的用户ID是12345”）。\n\n"
            f"对话内容：\n用户：{user_input}\nAI：{ai_response}\n\n"
            f"如果包含此类重要信息，请用一句话简洁地提取该事实；否则请回复“没有重要信息”。"
        )

        # 调用 LLM 来执行事实提取。
        extracted_fact = generate_text(
            "你是一名 fact-extraction 专家。",
            fact_extraction_prompt
        )

        # 检查 LLM 的响应是否表明发现了重要事实。
        if "no important fact" not in extracted_fact.lower():
            # 如果找到事实，打印调试消息并将其添加到记忆 token 列表中。
            print(f"--- [记忆增强：创建了新的记忆 token: '{extracted_fact}'] ---")
            self.memory_tokens.append(extracted_fact)

    def get_context(self, query: str) -> str:
        """
        通过将短期、最近对话与长期、持久的记忆 token 列表组合来构建上下文。
        """
        # 从短期滑动窗口获取上下文。
        recent_context = self.recent_memory.get_context(query)
        # 将记忆 token 列表格式化为可读的字符串。
        memory_token_context = "\n".join([f"- {token}" for token in self.memory_tokens])
        # 返回组合上下文，清楚地将长期事实与近期聊天分开。
        return f"### 关键记忆 token（长期事实）:\n{memory_token_context}\n\n### 近期对话:\n{recent_context}"

    def clear(self):
        """重置短期记忆和记忆 token 列表。"""
        self.recent_memory.clear()
        self.memory_tokens = []
        print("记忆增强记忆已清空。")

#### 记忆增强记忆演示

**观察重点：** 我们将在第一轮提到一个关键的长期偏好。 Agent 应该识别这是"关键记忆"并创建一个记忆 token。 在几轮更多对话将原始消息推出最近聊天窗口后，Agent 应该仍然能够通过读取其记忆 token 来回忆起该偏好。

In [ ]:
# 初始化 MemoryAugmentedMemory（记忆增强型记忆），窗口大小为2
# 这意味着短期记忆只保留最近两轮对话
mem_aug_memory = MemoryAugmentedMemory(window_size=2)

# 创建 AIAgent，并注入记忆增强策略
agent = AIAgent(memory_strategy=mem_aug_memory)

# --- 开始对话 ---

# 第一轮：用户提供一个需要长期记忆的重要信息
# 该信息应被事实抽取机制识别，并生成“记忆 token”
agent.chat("请记住这一点，在所有后续交互中：我对花生严重过敏。")

# 第二轮：普通对话
agent.chat("好，我们聊聊食谱。今晚有什么不错的晚餐建议？")

# 第三轮：继续对话
# 这一轮会把第一轮（过敏信息）从短期滑动窗口中挤出
agent.chat("不错，那甜点有什么推荐？")

# --- 测试记忆增强能力 ---

# 关键测试：
# 此时“花生过敏”已经不在短期上下文中
# Agent 唯一获取该信息的方式是长期记忆 token

agent.chat("你能推荐一个泰式绿咖喱的食谱吗？请确保对我是安全的。")

# 一个好的 Agent 应该：
# - 从长期记忆中检索“花生过敏”
# - 识别泰式咖喱可能含花生风险
# - 给出安全提醒或替代建议

# 清理记忆，用于下一次演示
mem_aug_memory.clear()

### 策略 6：分层记忆

| **最佳用于**              | **权衡**                                              |
| ----------------------------- | ------------------------------------------------------ |
| 多任务、具有不同信息类型的复杂 Agent          | 复杂的管理逻辑                           |

**理论：** 这是一个复合策略，模拟人类记忆在不同层次的工作方式。 它将多个更简单的记忆策略组合成一个层次结构。 常见设置是：
- **级别 1（工作记忆）：** 用于快速、即时上下文的 `SlidingWindowMemory`。
- **级别 2（长期记忆）：** 用于存储重要、持久事实的 `RetrievalMemory`。

关键是**提升**信息从 L1 到 L2 的逻辑。 我们的实现将使用启发式方法：如果对话轮次看起来特别重要（例如，包含"偏好"或"规则"等关键词），它会被添加到工作记忆和长期存储中。

In [ ]:
# --- 策略 6: 分层记忆 ---
# 此策略将多种记忆类型组合成一个更复杂的分层系统，
# 模拟人类记忆分为短期（工作）和长期存储的方式。
class HierarchicalMemory(BaseMemoryStrategy):
    def __init__(self, window_size: int = 2, k: int = 2, embedding_dim: int = 3584):
        """
        初始化分层记忆系统。

        参数:
            window_size: 短期工作记忆的大小（以交互次数计）。
            k: 从长期记忆检索的文档数量。
            embedding_dim: 长期记忆嵌入向量的维度。
        """
        print("正在初始化分层记忆...")
        # 级别 1：使用滑动窗口的快速、短期工作记忆。
        self.working_memory = SlidingWindowMemory(window_size=window_size)
        # 级别 2：使用检索系统的较慢、持久的长期记忆。
        self.long_term_memory = RetrievalMemory(k=k, embedding_dim=embedding_dim)
        # 一个简单的启发式方法：触发从工作记忆升级到长期记忆的关键词。
        self.promotion_keywords = ["记住", "规则", "偏好", "always", "never", "allergic"]

    def add_message(self, user_input: str, ai_response: str):
        """
        将消息添加到工作记忆，并根据其内容有条件地提升到长期记忆。
        """
        # 所有交互都被添加到快速、短期的工作记忆中。
        self.working_memory.add_message(user_input, ai_response)

        # 提升逻辑：检查用户的输入是否包含暗示信息重要的关键词，
        # 这些信息应该被长期存储。
        if any(keyword in user_input.lower() for keyword in self.promotion_keywords):
            print(f"--- [分层记忆：将消息提升到长期存储。] ---")
            # 如果找到关键词，也将交互添加到长期检索记忆中。
            self.long_term_memory.add_message(user_input, ai_response)

    def get_context(self, query: str) -> str:
        """
        通过组合来自长期和短期记忆层的有相关信息来构建丰富的上下文。
        """
        # 从工作记忆检索最近的对话。
        working_context = self.working_memory.get_context(query)
        # 根据当前查询从长期记忆检索语义相关的事实。
        long_term_context = self.long_term_memory.get_context(query)

        # 组合两个上下文，清楚标记它们的来源以供 LLM 参考。
        return f"### 从长期记忆检索到的内容:\n{long_term_context}\n\n### 近期对话（工作记忆）:\n{working_context}"

    def clear(self):
        """重置工作和长期记忆存储。"""
        self.working_memory.clear()
        self.long_term_memory.clear()
        print("分层记忆已清空。")

#### 分层记忆演示

**观察重点：** 我们将使用关键词"记住"来陈述一个偏好。 这将触发消息被保存在长期 `RetrievalMemory` 中。 在几轮将消息推出短期 `SlidingWindowMemory` 后，我们将询问一个相关问题。 Agent 应该通过从其长期存储中检索来成功回答。

In [ ]:
# 初始化分层记忆（HierarchicalMemory）
# 它结合了短期滑动窗口 + 长期检索系统
hierarchical_memory = HierarchicalMemory()

# 创建 AI Agent，并注入分层记忆策略
agent = AIAgent(memory_strategy=hierarchical_memory)

# --- 开始对话 ---

# 第一轮：用户提供重要信息（包含关键词 "remember"）
# 这会触发“记忆提升机制”，将信息同时写入短期和长期记忆
agent.chat("请记住我的用户ID是 AX-7890。")

# 第二轮：普通对话内容，只进入短期记忆
agent.chat("我们聊聊天气吧，今天非常晴朗。")

# 第三轮：另一个普通话题，会把最早的用户ID从短期窗口挤出
agent.chat("我打算等会去散步。")

# --- 测试分层记忆检索 ---

# 此时 User ID 已经不在短期工作记忆中（被滑动窗口移除）
# Agent 必须依赖长期记忆进行检索
agent.chat("我要登录我的账号，你能告诉我我的ID吗？")

# 一个成功的 Agent 应该能够从长期记忆中检索到 AX-7890
# 因为第一句包含 "remember"，触发了长期存储机制

# 清理记忆，为下一次演示做准备
hierarchical_memory.clear()

### 策略 7：基于图的记忆

| **最佳用于**              | **权衡**                                              |
| ----------------------------- | ------------------------------------------------------ |
| 事实之间关系重要的系统              | 最适合结构化知识，维护需要更多努力         |

**理论：** 此策略超越了存储非结构化文本。 它将信息表示为**知识图谱**，由节点（实体）和边（关系）组成。 For example, `(Sam) -[WorksFor]-> (Innovatech) -[FocusesOn]-> (Energy)`.

这对于回答需要推理连接的复杂查询非常强大。 主要挑战是填充图。 我们将使用一种强大技术：**使用 LLM 作为工具**从非结构化对话文本中提取结构化的 `(主语、关系、宾语)`（`(subject, relation, object)`） 三元组。

In [ ]:
# 导入图数据结构和正则表达式所需的库。
import networkx as nx
import re

# --- 策略 7: 基于图的记忆 ---
# 此策略将信息表示为结构化知识图谱，由节点（实体如 'Sam'、'Innovatech'）
# 和边（关系如 'works_for'、'focuses_on'）组成。
# 它使用 LLM 本身从非结构化对话文本中提取这些结构化的
# 三元组（subject、relation、object）。这是"LLM 作为工具"的一种形式。
class GraphMemory(BaseMemoryStrategy):
    def __init__(self):
        """使用空的 NetworkX 有向图初始化记忆。"""
        # DiGraph 适合表示有向关系（例如，Sam -> works_for -> Innovatech）。
        self.graph = nx.DiGraph()

    def _extract_triples(self, text: str) -> list[tuple[str, str, str]]:
        """
        使用 LLM 从给定文本中提取知识三元组（主语、关系、宾语）。
        这是"LLM 作为工具"的一种形式，其中模型的自然语言理解能力
        被用于创建结构化数据。
        """
        print("--- [图记忆：尝试从文本中提取三元组。] ---")
        # 构建一个详细的提示词，说明 LLM 的角色和所需的输出格式。
        # 提供清晰的示例对于获得可靠的、结构化的输出至关重要。
        extraction_prompt = (
            f"你是一个 knowledge extraction engine。你的任务是从给定文本中抽取“主语-关系-宾语”（SPO）三元组。"
            f"请严格按照 Python 元组列表的格式输出结果。例如："
            f"[('Sam', 'works_for', 'Innovatech'), ('Innovatech', 'focuses_on', 'Energy')]。"
            f"如果没有发现任何三元组，请返回空列表 []。\n\n"
            f"待分析文本：\n\"\"\"{text}\"\"\""
        )

        # 使用专门的提示词调用 LLM。
        response_text = generate_text("你是一个专业的 knowledge graph extractor.", extraction_prompt)

        # 安全地解析 LLM 响应中的元组列表字符串表示。
        try:
            # 使用正则表达式比使用 `eval()` 更安全，因为它避免了
            # 执行 LLM 输出中可能恶意或意外包含的任意代码。
            # 此正则表达式查找匹配 ('item1', 'item2', 'item3') 的模式。
            found_triples = re.findall(r"\(['\"](.*?)['\"],\s*['\"](.*?)['\"],\s*['\"](.*?)['\"]\)", response_text)
            print(f"--- [图记忆：提取的三元组: {found_triples}] ---")
            return found_triples
        except Exception as e:
            # 如果解析失败，记录错误并返回空列表以防止崩溃。
            print(f"无法从 LLM 响应中解析三元组: {e}")
            return []

    def add_message(self, user_input: str, ai_response: str):
        """从最新对话交互中提取三元组并将它们添加到知识图谱中。"""
        # 合并用户和 AI 消息以提供完整的提取上下文。
        full_text = f"User: {user_input}\nAI: {ai_response}"
        # 调用辅助方法来获取结构化的三元组。
        triples = self._extract_triples(full_text)
        # 遍历提取的三元组。
        for subject, relation, obj in triples:
            # 向图添加一条边。`add_edge` 会自动创建尚不存在的节点
            #（主语、宾语）。关系存储为边属性。
            # .strip() 删除任何前导/尾随空白以获得更清晰的数据。
            self.graph.add_edge(subject.strip(), obj.strip(), relation=relation.strip())

    def get_context(self, query: str) -> str:
        """
        通过在图中查找查询中的实体来检索上下文，
        并返回它们所有已知的关系。
        """
        # 如果图为空，则无法提供上下文。
        if not self.graph.nodes:
            return "知识图谱为空。"

        # 这是一种简单的实体链接方法：它大写查询中的单词并检查
        # 它们是否作为节点存在于图中。更高级的系统将使用
        # 自然语言处理（NLP）来更准确地识别命名实体。
        query_entities = [word.capitalize() for word in query.replace('?', '').split() if word.capitalize() in self.graph.nodes]

        # 如果在图谱中找不到查询中的任何实体，则无法提供特定上下文。
        if not query_entities:
            return "在知识图谱中找不到与您的查询相关的实体。"

        context_parts = []
        # 使用 set() 来确保每个唯一实体只处理一次。
        for entity in set(query_entities):
            # 查找所有出边（例如，Sam -> works_for -> X）
            for u, v, data in self.graph.out_edges(entity, data=True):
                context_parts.append(f"{u} --[{data['relation']}]--> {v}")
            # 查找所有入边（例如，X -> is_located_in -> New York）
            for u, v, data in self.graph.in_edges(entity, data=True):
                context_parts.append(f"{u} --[{data['relation']}]--> {v}")

        # 将检索到的事实组合成单个上下文字符串，删除重复项并排序以保持一致性。
        return "### 从知识图谱中检索到的事实:\n" + "\n".join(sorted(list(set(context_parts))))

    def clear(self):
        """通过清除图中的所有节点和边来重置记忆。"""
        self.graph.clear()
        print("图记忆已清空。")

#### 基于图的记忆演示

**观察重点：** 当我们聊天时，Agent 将调用 LLM 提取三元组并构建其知识图谱。 你将看到 `[提取的三元组]` 调试消息。 当我们最终提问时，Agent 将通过显示其学习到的结构化关系来提供上下文，允许它回答关于不同实体如何连接的问题。

In [ ]:
# 初始化图结构记忆策略（GraphMemory）
graph_memory = GraphMemory()

# 创建 AI Agent，并注入图记忆策略
agent = AIAgent(memory_strategy=graph_memory)

# --- 开始对话 ---

# 第一轮：Agent 将尝试抽取三元组 ('Clara', 'works_for', 'FutureScape')
agent.chat("一个叫 Clara 的人为一家名为 'FutureScape' 的公司工作。")

# 第二轮：Agent 将尝试抽取三元组 ('FutureScape', 'is_based_in', 'Berlin')
agent.chat("FutureScape 总部位于柏林。")

# 第三轮：Agent 将尝试抽取三元组 ('Clara', 'main_project_is', 'Odyssey')
agent.chat("Clara 的主要项目名为 'Odyssey'。")

# --- 测试图推理能力 ---

# 现在提出一个需要关联推理的问题
# Agent 会识别出查询中的实体 'Clara'
# 然后在图中查找与 Clara 相关的所有关系
# 并将这些结构化信息作为上下文提供给 LLM
agent.chat("告诉我关于 Clara 的项目。")

# 一个成功的 Agent 应该能够利用已抽取的事实
# 推断 Clara 与她的项目之间的关系

# 清理记忆，为下一次演示做准备
graph_memory.clear()

### 策略 8：压缩与整合记忆

| **最佳用于**              | **权衡**                                              |
| ----------------------------- | ------------------------------------------------------ |
| 可扩展的低成本记忆              | 存在有损或过于抽象回忆的风险                |

**理论：** 这是一种更明确、更积极的摘要形式。 目标不是创建叙事摘要，而是将每条信息压缩成其最密集的事实表示。 把它想象成将冗长的段落转换为几个要点。

我们的实现将获取每个对话轮次，并使用 LLM 将其重写为简洁的压缩事实。 记忆只是一个这些压缩事实的列表。 与存储完整文本相比，这可以节省大量 token，但丢失细微差别的风险比标准摘要更高。

In [ ]:
# --- 策略 8: 压缩与整合记忆 ---
# 此策略是一种更明确、更积极的摘要形式。
# 目标不是创建叙事摘要，而是将每条信息压缩成其最密集的事实表示。
# 可以把它想象成将冗长的段落转换为几个要点。
class CompressionMemory(BaseMemoryStrategy):
    def __init__(self):
        """初始化压缩记忆，使用空列表存储压缩后的事实。"""
        self.compressed_facts = []

    def add_message(self, user_input: str, ai_response: str):
        """
        使用 LLM 将对话交互压缩为密集的事实。
        """
        # 要压缩的文本 = 用户输入 + AI 响应
        text_to_compress = f"User: {user_input}\nAI: {ai_response}"

       # 压缩提示词：要求 LLM 将文本压缩为最核心的事实陈述
        compression_prompt = (
            f"你是一个数据压缩引擎。你的任务是将以下文本压缩为最核心、最本质的事实陈述。"
            f"尽可能简洁，去除所有对话性冗余信息。"
            f"例如：'用户询问了我的名字，而我作为AI助手回答我的名字是AI助手' 应压缩为 '用户询问AI的名字'。\n\n"
            f"待压缩文本：\n\"\"\"{text_to_compress}\"\"\""
        )

        # 调用 LLM 执行压缩
        compressed_fact = generate_text("你是一个专业的数据压缩引擎", compression_prompt)
        print(f"--- [压缩记忆：存储新事实: '{compressed_fact}'] ---")
        self.compressed_facts.append(compressed_fact)

    def get_context(self, query: str) -> str:
        """
        返回所有压缩事实的列表。
        """
        if not self.compressed_facts:
            return "记忆中没有任何压缩事实。"

        return "### 压缩的事实记忆:\n- " + "\n- ".join(self.compressed_facts)

    def clear(self):
        """通过清空压缩事实列表来重置记忆。"""
        self.compressed_facts = []
        print("压缩记忆已清空。")

#### 压缩与整合记忆演示

**观察重点：** 每轮后会出现一个 `[压缩记忆：存储新事实]` 消息，显示高度凝练的交互版本。 最终发送给 LLM 的上下文将是一个简洁事实的项目符号列表，比原始对话更节省 token。

In [ ]:
compression_memory = CompressionMemory()
agent = AIAgent(memory_strategy=compression_memory)

agent.chat("好的，我已经决定会议场地了。将使用“Metropolitan Convention Center”。")
agent.chat("日期已确认：2025年10月26日。")
agent.chat("请帮我总结一下这次会议计划的关键细节。")

compression_memory.clear()

### 策略 9：类操作系统记忆管理（概念模拟）

| **最佳用于**              | **权衡**                                              |
| ----------------------------- | ------------------------------------------------------ |
| 具有动态记忆需求的大规模系统            | 概念强大，架构复杂                           |

**理论：** 这个高级概念借鉴了计算机操作系统如何管理内存。 它将 LLM 的上下文窗口视为**RAM**（快速、小、昂贵）和外部存储视为**硬盘**（慢、大、便宜）。 信息在它们之间移动。
- **换页出：** 当 RAM（活跃上下文）满时，策略（如最近最少使用 - LRU）将最旧的信息移动到硬盘（被动存储）。
- **换页入（缺页异常）：** 当查询需要不在 RAM 中的信息时，发生"缺页异常"。 然后系统从硬盘检索（换页入）所需信息回 RAM，可能交换其他内容。

**我们的模拟：** 我们将创建一个 `active_memory`（列表）和 `passive_memory`（字典用于快速查找）。 当 `active_memory` 满时，我们将 LRU 项移动到 `passive_memory`。 我们的 `get_context` 将使用检索来查看查询是否需要来自 `passive_memory` 的任何数据，模拟换页入。

In [ ]:
# --- 策略 9: 类操作系统记忆管理（模拟）---
# 这个高级概念借鉴了计算机操作系统如何管理内存。
# 它将 LLM 的上下文窗口视为 RAM（快速、小、昂贵），
# 外部存储视为硬盘（慢、大、便宜）。信息在它们之间移动。
#
# - 换页出：当 RAM（活跃上下文）满时，策略（如最近最少使用 - LRU）
#   将最旧的信息移动到硬盘（被动存储）。
# - 换页入（缺页异常）：当查询需要不在 RAM 中的信息时，发生"缺页异常"。
#   然后系统从硬盘检索（换页入）所需信息回 RAM，可能交换其他内容。
class OSMemory(BaseMemoryStrategy):
    def __init__(self, ram_size: int = 2):
        """
        初始化类操作系统记忆。

        参数:
            ram_size: 活跃记忆（RAM）中的最大交互次数。
        """
        self.ram_size = ram_size  # RAM 中最大交互次数
        self.active_memory = deque()  # 我们的 'RAM'
        self.passive_memory = {}  # 我们的'硬盘'，一个键值存储
        self.turn_count = 0

    def add_message(self, user_input: str, ai_response: str):
        """
        将交互添加到活跃记忆，如果 RAM 已满则换页到被动存储。
        """
        turn_id = self.turn_count
        turn_data = f"User: {user_input}\nAI: {ai_response}"

        # 如果活跃记忆已满，则将最近最少使用（最旧）的项目换页到被动存储
        if len(self.active_memory) >= self.ram_size:
            # 从 RAM 中移除最旧的交互
            lru_turn_id, lru_turn_data = self.active_memory.popleft()
            # 存储到被动存储（硬盘）
            self.passive_memory[lru_turn_id] = lru_turn_data
            print(f"--- [OS 内存：将交互 {lru_turn_id} 换页出到被动存储。] ---")

        # 将新交互添加到活跃记忆（RAM）
        self.active_memory.append((turn_id, turn_data))
        self.turn_count += 1

    def get_context(self, query: str) -> str:
        """
        提供 RAM 上下文，并模拟"缺页异常"从被动存储中拉取数据。
        """
        # 从活跃记忆构建上下文
        active_context = "\n".join([data for _, data in self.active_memory])

        # 模拟缺页异常：检查查询是否与被动存储中的某些内容更相似
        # 在真实系统中，这将使用嵌入进行相似性搜索。
        # 对于此演示，我们将进行简单的关键词搜索。
        paged_in_context = ""
        for turn_id, data in self.passive_memory.items():
            # 检查查询中的关键词是否出现在被动存储的数据中
            if any(word in data.lower() for word in query.lower().split() if len(word) > 3):
                paged_in_context += f"\n(从交互 {turn_id} 换页入): {data}"
                print(f"--- [OS 内存：缺页异常！从被动存储换页入交互 {turn_id}。] ---")

        return f"### 活跃记忆 (RAM):\n{active_context}\n\n### 从被动存储（磁盘）换页入:\n{paged_in_context}"

    def clear(self):
        """重置所有记忆存储。"""
        self.active_memory.clear()
        self.passive_memory = {}
        self.turn_count = 0
        print("类操作系统记忆已清空。")

#### 类操作系统记忆演示

**观察重点：** 我们将进行一个对话，其中第一轮包含一个密钥代码。 当我们继续时，你会看到一个 `[换页出]` 消息，因为这轮被从"RAM"移动到"磁盘"。 当我们最终询问密钥代码时，将触发一个 `[缺页异常！]`，Agent 将"换页入"该特定记忆来回答问题。

In [ ]:
# 初始化操作系统式记忆（OSMemory）
# ram_size=2 表示最多只能同时保存 2 条记忆（模拟有限内存）
os_memory = OSMemory(ram_size=2)

# 创建 AI Agent，并注入该记忆策略
agent = AIAgent(memory_strategy=os_memory)

# --- 开始对话 ---

# 第一条信息：关键秘密（启动代码）
agent.chat("启动代码是 'Orion-Delta-7'。")

# 第二条信息：天气信息
agent.chat("发射当天的天气看起来很好。")

# 第三条信息：发射时间
# 此时内存已满（2条），会触发页面置换（page out）
# 最早或最不常用的记忆可能被淘汰（例如启动代码）
agent.chat("发射窗口将在世界时 0400 开启。")

# --- 测试已被换出的记忆 ---

# 查询已经被“换出内存”的信息
agent.chat("我需要确认发射代码是多少。")

# 如果实现正确：
# Agent 将无法从当前内存中找到启动代码（因为已被 page out）

# 清理内存，用于下一次测试
os_memory.clear()

## 实验总结与最终思考

恭喜你完成了对 AI Agent 记忆的深度探索！ 我们已成功实现并测试了九种不同的策略，亲眼见证了每种策略如何影响 Agent 的性能、成本和能力。

**本实验的关键收获：**

1. **没有银弹：** 记忆的选择从根本上说是基于 Agent 目的的设计决策。 简单的问答机器人可能只需要 `SlidingWindow`，而长期个人助理将在 `Hierarchical` 或 `Retrieval` 系统中茁壮成长。

2. **复杂性的范围：** 我们看到了从简单的线性存储（`Sequential`）到复杂的结构化数据（`GraphMemory`）和动态系统（`OSMemory`）的清晰进展。 增加复杂性带来更多力量，但需要更复杂的工程。

3. **LLM 作为工具：** 几种高级策略（`Summarization`、`GraphMemory`、`MemoryAugmented`）不只是将 LLM 用于聊天；它们将它用作处理、构建和压缩记忆本身的智能工具。 这是现代 Agent 设计中的强大范式。

4. **混合系统是未来：** 最稳健的生产 Agent 通常使用混合方法。 我们的 `HierarchicalMemory` 是一个典型例子，结合了滑动窗口的速度和检索的精确度。 你可以混合匹配这些策略来创建适合你精确需求的定制记忆架构。

### 下一步

本笔记本是一个起点。你可以通过以下方式继续探索：
- **调优参数：** 调整 `window_size`、`k` 用于检索，以及摘要阈值，看看它们如何影响性能。
- **实现更高级的逻辑：** 使用嵌入改进 `HierarchicalMemory` 中的提升逻辑或 `OSMemory` 中的缺页异常检测。
- **创建新的混合体：** 将 `GraphMemory` 与 `RetrievalMemory` 组合，同时搜索结构和非结构化数据。

感谢你加入本实验。快乐构建！